# Example 11.2 / 1: A Smooth Resource-Allocation Model

Companion notebook for **11.2 Nonlinear Models** in the AMPL lab Documentation, section "A Smooth Resource-Allocation Model". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## A Smooth Resource-Allocation Model

Suppose returns exhibit diminishing marginal benefit. With positive coefficients and `log(1 + Spend[p])`, the objective is concave over `Spend[p] >= 0`. Maximizing it subject to linear constraints is a convex optimization problem.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROJECTS;                  # candidate projects
param benefit {PROJECTS} > 0;  # benefit weight per project
param budget > 0;              # total spending available

# spending per project; start from an even split
var Spend {PROJECTS} >= 0, := budget / card(PROJECTS);

maximize TotalBenefit:  # concave: diminishing returns via log
    sum {p in PROJECTS} benefit[p] * log(1 + Spend[p]);

subject to Budget:  # total spending within budget
    sum {p in PROJECTS} Spend[p] <= budget;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROJECTS"] = ["efficiency", "quality", "capacity"]
ampl.param["benefit"] = {"efficiency": 5, "quality": 3, "capacity": 2}
ampl.param["budget"] = 12

In [5]:
# run.py
ampl.solve(solver="ipopt", verbose=False)
print(ampl.solve_result)
print(ampl.var["Spend"].to_dict())
print(round(ampl.con["Budget"].slack(), 6))

solved
{'capacity': 2.0000000259270365, 'efficiency': 6.500000055049701, 'quality': 3.500000035261875}
-0.0


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
set PROJECTS := efficiency quality capacity;
param benefit := efficiency 5 quality 3 capacity 2;
param budget := 12;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver ipopt;
solve > ipopt.log;
display Spend, Budget.slack;

 
Ipopt 3.12.13: Optimal Solution Found

suffix ipopt_zU_out OUT;
suffix ipopt_zL_out OUT;
Spend [*] :=
  capacity  2
efficiency  6.5
   quality  3.5
;

Budget.slack = -1.16239e-07



### Another way in amplpy

The two `suffix` lines are AMPL announcing the bound multipliers that Ipopt returns. The
pure-AMPL slack shows that the budget is exceeded by about $10^{-7}$, which is feasible only
within the solver's tolerance. Recomputing the domain margin and the budget residual from the
returned values, with the same `model.mod` and `data.py`, makes that check explicit:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
ampl.set["PROJECTS"] = ["efficiency", "quality", "capacity"]
ampl.param["benefit"] = {"efficiency": 5, "quality": 3, "capacity": 2}
ampl.param["budget"] = 12

In [11]:
# run.py
ampl.solve(solver="ipopt", verbose=False)
spend = ampl.var["Spend"].to_dict()
print("smallest log argument:", round(min(1 + v for v in spend.values()), 6))
print(f"budget excess: {sum(spend.values()) - 12:.1e}")

smallest log argument: 3.0
budget excess: 1.2e-07
